<a href="https://colab.research.google.com/github/nitshar002/real-estate-pricing-model/blob/main/Modeling_CaliforniaRealEstateProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Importing Advanced Preprocessing File

In [ ]:
user = 'Nitya' #This is whoever's advanced preprocessing you want to use.
user = user.lower()

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import pandas as pd
import numpy as np

train_path = f"/content/drive/MyDrive/IDX Housing Data/{user}_AdvancedProcessingAndModeling/{user}_train_processed.parquet"
test_path = f"/content/drive/MyDrive/IDX Housing Data/{user}_AdvancedProcessingAndModeling/{user}_test_processed.parquet"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

print(train_df.shape)
print(test_df.shape)

print(train_df.columns)

Mounted at /content/drive
(111588, 39)
(6928, 39)
Index(['ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'ClosePrice',
       'Latitude', 'Longitude', 'LivingArea', 'DaysOnMarket',
       'AttachedGarageYN', 'ParkingTotal', 'YearBuilt',
       'BathroomsTotalInteger', 'BedroomsTotal', 'FireplaceYN', 'Stories',
       'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'LotSizeSquareFeet', 'HasCarpet', 'HasVinyl', 'HasStone', 'HasBamboo',
       'HasConcrete', 'HasBrick', 'HasLaminate', 'HasTile', 'HasWood',
       'HasUnknownFlooring', 'Monthly_HOA', 'District_Avg_Price',
       'Postal_Code_Encoded', 'log_HOA', 'Home_Age', 'Bed_to_Bath',
       'Living_Area_to_Bedrooms', 'Living_Area_per_Story',
       'DistNearestRestaurantMi'],
      dtype='object')


### Pre Modeling

In [ ]:
## Split features
X_train = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']
y_train_log = np.log1p(train_df['ClosePrice'])

X_test = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']
y_test_log = np.log1p(test_df['ClosePrice'])

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = train_df.drop(columns=['ClosePrice']).select_dtypes(include=['number']).columns

binary_cols = [
    c for c in num_cols
    if train_df[c].dropna().isin([0,1]).all()
]

scale_cols = [c for c in num_cols if c not in binary_cols]

# Replace inf/-inf with NaN first
train_df.replace([np.inf, -np.inf], np.nan, inplace=True)
test_df.replace([np.inf, -np.inf], np.nan, inplace=True)

scale_cols = [c for c in X_train.select_dtypes(include=['number']).columns if c != 'ClosePrice']
# Imputation
for c in scale_cols:
    median = train_df[c].median()
    X_test[c] = X_test[c].fillna(median)
'''
# Home_Age cannot be negative
X_test['Home_Age'] = X_test['Home_Age'].clip(lower=0)
'''
# Ratios like Bed_to_Bath and Living_Area_to_Bedrooms cannot be negative
for col in ['log_HOA','Living_Area_per_Story']:
    X_test[col] = X_test[col].clip(lower=0)

scaler_X = StandardScaler(copy=False)

train_df[scale_cols] = scaler_X.fit_transform(
    train_df[scale_cols].astype('float32')
)

test_df[scale_cols] = scaler_X.transform(
    test_df[scale_cols].astype('float32')
)

In [ ]:
# # TYLER:
# # Found that you accidentally didn't scale X_train or X_test. Nvm you did in the linear regression test

# X_train[scale_cols] = scaler_X.fit_transform(X_train[scale_cols].astype('float32'))

# # ONLY Transform on Test (No fitting!)
# X_test[scale_cols] = scaler_X.transform(X_test[scale_cols].astype('float32'))

# # Realized this doesn't matter for LightGBM. So maybe leave unscaled for simplicity?

# Modeling

## Linear regression test - R squared, MAPE and MdAPE

In [ ]:
unknown_cols = [col for col in train_df.columns if (train_df[col] == 'Unknown').any()]

print(f"Columns containing 'Unknown': {unknown_cols}")
# dropping HighSchoolDistrict for now.. to make sure everything is set up.
#train_df = train_df.drop(columns=['HighSchoolDistrict'])
#test_df = test_df.drop(columns=['HighSchoolDistrict'])

Columns containing 'Unknown': []


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# WITH LOG TRANSFORMATION

# Split features and target
X_train = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

# Align columns of X_test with X_train
# This is crucial because the error indicates a column mismatch (e.g., 'UnparsedAddress' in X_test but not X_train)
X_test = X_test[X_train.columns]

# Create model
lr_model = LinearRegression()

# Fit model
lr_model.fit(X_train, y_train_log)

# Predict
y_pred_log = lr_model.predict(X_test)
y_pred = np.expm1(y_pred_log)

# Metrics
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
mdape = np.median(np.abs((y_test - y_pred) / y_test))  # Manual MdAPE

print(f"R^2 Score: {r2:.4f}")
print(f"MAPE: {mape:.4f}")
print(f"MdAPE: {mdape:.4f}")

train_preds_log = lr_model.predict(X_train)
train_preds = np.expm1(train_preds_log)

train_r2 = r2_score(y_train, train_preds)
train_mape = mean_absolute_percentage_error(y_train, train_preds)
train_mdape = np.median(np.abs((y_train - train_preds) / y_train))

print(f"\nTraining R^2 Score: {train_r2:.4f}")
print(f"MAPE: {train_mape:.4f}")
print(f"MdAPE: {train_mdape:.4f}")

R^2 Score: 0.4794
MAPE: 0.2203
MdAPE: 0.1324

Training R^2 Score: 0.5252
MAPE: 0.1831
MdAPE: 0.1315


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create model
# Modified parameters a bit and achieved comparable R^2
rf_model = RandomForestRegressor(n_estimators=150, max_depth=20, n_jobs=-1, random_state=42)

# Fit model
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

# Metrics
r2_rf = r2_score(y_test, y_pred_rf)
mape_rf = mean_absolute_percentage_error(y_test, y_pred_rf)
mdape_rf = np.median(np.abs((y_test - y_pred_rf) / y_test))

print(f"Random Forest R^2 Score: {r2_rf:.4f}")
print(f"Random Forest MAPE: {mape_rf:.4f}")
print(f"Random Forest MdAPE: {mdape_rf:.4f}")

train_preds = rf_model.predict(X_train)

train_r2 = r2_score(y_train, train_preds)
train_mape_rf = mean_absolute_percentage_error(y_train, train_preds)
train_mdape_rf = np.median(np.abs((y_train - train_preds) / y_train))

print(f"\nTraining R^2 Score: {train_r2:.4f}")
print(f"Training Random Forest MAPE: {train_mape_rf:.4f}")
print(f"Training Random Forest MdAPE: {train_mdape_rf:.4f}")

KeyboardInterrupt: 

# XGBoost

In [ ]:
import xgboost
from xgboost import XGBRegressor

# Create model
xgb_model = XGBRegressor(
    n_estimators=250,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)

# Fit model
xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)

# Metrics
r2_xgb = r2_score(y_test, y_pred_xgb)
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)
mdape_xgb = np.median(np.abs((y_test - y_pred_xgb) / y_test))

print(f"XGBoost R^2 Score: {r2_xgb:.4f}")
print(f"XGBoost MAPE: {mape_xgb:.4f}")
print(f"XGBoost MdAPE: {mdape_xgb:.4f}")

train_preds = xgb_model.predict(X_train)

train_r2 = r2_score(y_train, train_preds)
train_mape_xgb = mean_absolute_percentage_error(y_train, train_preds)
train_mdape_xgb = np.median(np.abs((y_train - train_preds) / y_train))

print(f"\nTraining R^2 Score: {train_r2:.4f}")
print(f"Training XGBoost MAPE: {train_mape_xgb:.4f}")
print(f"Training XGBoost MdAPE: {train_mdape_xgb:.4f}")

# Neural Network

In [ ]:
# Worse than random forest and XGBoost but better than linear regression

from sklearn.neural_network import MLPRegressor

# Create model
nn_model = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    max_iter=200,
    random_state=42
)

# Fit model
nn_model.fit(X_train, y_train)

# Predict
y_pred_nn = nn_model.predict(X_test)

# Metrics
r2_nn = r2_score(y_test, y_pred_nn)
mape_nn = mean_absolute_percentage_error(y_test, y_pred_nn)
mdape_nn = np.median(np.abs((y_test - y_pred_nn) / y_test))

print(f"Neural Network R^2 Score: {r2_nn:.4f}")
print(f"Neural Network MAPE: {mape_nn:.4f}")
print(f"Neural Network MdAPE: {mdape_nn:.4f}")

train_preds = nn_model.predict(X_train)

train_r2 = r2_score(y_train, train_preds)
train_mape_nn = mean_absolute_percentage_error(y_train, train_preds)
train_mdape_nn = np.median(np.abs((y_train - train_preds) / y_train))

print(f"\nTraining R^2 Score: {train_r2:.4f}")
print(f"Training Neural Network MAPE: {train_mape_nn:.4f}")
print(f"Training Neural Network MdAPE: {train_mdape_nn:.4f}")

# LightGBM

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Create model
lgb_model = LGBMRegressor(
    n_estimators=300,
    max_depth=8,
    num_leaves=64,
    learning_rate=0.1,
    subsample=0.8,          # same as XGBoost subsample
    colsample_bytree=0.8,   # same as XGBoost colsample_bytree
    n_jobs=-1,
    random_state=42
)

# Give houses over $5M a heavier weight (e.g., 1.5x the penalty for getting it wrong)
# y_train is training target (before log transformation)
# weights = np.where(y_train > 5000000, 1.5, 1.0)

# Pass the weights into the fit method
# lgb_model.fit(X_train, y_train_log, sample_weight=weights)

# # Fit model
lgb_model.fit(X_train, y_train_log)

# Predict
y_pred_lgb_log = lgb_model.predict(X_test)
y_pred_lgb = np.expm1(y_pred_lgb_log)

# Metrics (Test)
r2_lgb = r2_score(y_test, y_pred_lgb)
mape_lgb = mean_absolute_percentage_error(y_test, y_pred_lgb)
mdape_lgb = np.median(np.abs((y_test - y_pred_lgb) / y_test))
mae_lgb = mean_absolute_error(y_test, y_pred_lgb)

print(f"LightGBM R^2 Score: {r2_lgb:.4f}")
print(f"LightGBM MAPE: {mape_lgb:.4f}")
print(f"LightGBM MdAPE: {mdape_lgb:.4f}")
print(f"LightGBM MAE: {mae_lgb:,.0f}")

# Training performance
train_preds_log = lgb_model.predict(X_train)
train_preds = np.expm1(train_preds_log)

train_r2 = r2_score(y_train, train_preds)
train_mape = mean_absolute_percentage_error(y_train, train_preds)
train_mdape = np.median(np.abs((y_train - train_preds) / y_train))

print(f"\nTraining R^2 Score: {train_r2:.4f}")
print(f"Training LightGBM MAPE: {train_mape:.4f}")
print(f"Training LightGBM MdAPE: {train_mdape:.4f}")

# Results between training and test set show little overfitting!!

Start Tyler

Exporting your model (the BEST model) using pkl

In [ ]:
# print(lgb_model.feature_names_in_, '\n')

# 1. Create X sample of first 10 rows in X_test
X_sample = X_test[lgb_model.feature_names_in_]

# 2. Generate the predictions
predicted_prices_log = lgb_model.predict(X_sample)
predicted_prices = np.expm1(predicted_prices_log)

# 3. Grab the actual true prices
actual_prices = y_test.values

# 4. Calculate the Absolute Percentage Error for each row
error_pct = np.abs(predicted_prices - actual_prices) / actual_prices

# 5. Put them side-by-side in a clean DataFrame
results_df = pd.DataFrame({
    'Actual Price': actual_prices,
    'Predicted Price': predicted_prices,
    'Difference ($)': predicted_prices - actual_prices,
    'Error (%)': error_pct
})

# 6. Format the numbers to look like real currency and percentages
formatted_results = results_df.head(10).style.format({
    'Actual Price': '${:,.0f}',
    'Predicted Price': '${:,.0f}',
    'Difference ($)': '${:,.0f}',
    'Error (%)': '{:.2%}'
})

# Display the table
formatted_results

In [ ]:
under_5_pct = (results_df['Error (%)'] <= 0.05).mean() * 100
under_10_pct = (results_df['Error (%)'] <= 0.10).mean() * 100
under_20_pct = (results_df['Error (%)'] <= 0.20).mean() * 100
over_70_pct = (results_df['Error (%)'] >= 0.70).mean() * 100
over_50_pct = (results_df['Error (%)'] >= 0.50).mean() * 100

print(f"Predictions within 5% error: {under_5_pct:.1f}%")
print(f"Predictions within 10% error: {under_10_pct:.1f}%")
print(f"Predictions within 20% error: {under_20_pct:.1f}%")

print()
print(f"Predictions over 50% error: {over_50_pct:.1f}%")
print(f"Predictions over 70% error: {over_70_pct:.1f}%")

In [ ]:
plt.figure(figsize=(8, 8))
sns.scatterplot(x='Actual Price', y='Predicted Price', data=results_df, alpha=0.6)

# Draw the "Perfect Prediction" diagonal line
max_val = max(results_df['Actual Price'].max(), results_df['Predicted Price'].max())
plt.plot([0, max_val], [0, max_val], color='red', linestyle='--')

plt.title('Actual vs. Predicted Housing Prices')
plt.xlabel('Actual Price ($ in Millions)')
plt.ylabel('Predicted Price ($ in Millions)')
plt.show()

In [ ]:
# Grab a row from our 10-row sample
idx = 1
first_house = X_sample.iloc[idx]

print("--- Data for the First House ---")
print(f"Actual Price: ${actual_prices[idx]:,.0f}")
print(f"Predicted Price: ${predicted_prices[idx]:,.0f}")
print(f"Error: {error_pct[idx]:.2%}\n")

# Convert the single row into a vertical DataFrame so it's easy to read
pd.DataFrame({'Feature Value': first_house})

Exporting Model

In [ ]:
# Actual exportation
from pathlib import Path
import joblib

# 1. Define your save directory
save_dir = Path(f"/content/drive/MyDrive/IDX Housing Data/group_AdvancedProcessingAndModeling/models/")

# 2. Create the directory if it doesn't exist
save_dir.mkdir(parents=True, exist_ok=True)

# 3. Dump the model
model_name = lgb_model
joblib.dump(model_name, save_dir / 'lgb_model_zengtao_v1.pkl')

# 4. Dump the scaler object
joblib.dump(scaler_X, save_dir / 'scaler_X.pkl')
# 5. Dump the columns that scaler was applied to
joblib.dump(scale_cols, save_dir / 'scale_cols.pkl')

print(f"Model and scalers saved successfully to: {save_dir.absolute()}")

End Tyler